In [1]:
import os, requests
os.makedirs('/content/data/raw', exist_ok=True)
url = 'https://raw.githubusercontent.com/Nayak-D/FlyRank---Internship/main/data/raw/content_refresh_anonymized.csv'
print('Downloading', url)
resp = requests.get(url, stream=True, timeout=60)
resp.raise_for_status()
with open('/content/data/raw/content_refresh_anonymized.csv','wb') as f:
    for chunk in resp.iter_content(1024*1024):
        if chunk:
            f.write(chunk)
print('Downloaded to /content/data/raw/content_refresh_anonymized.csv')


Downloaded to /content/data/raw/content_refresh_anonymized.csv


# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nayak-D/FlyRank---Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This lane is best framed as a scoring / ranking task. The work should score each page for refresh priority and let a reviewer sort pages by which ones are most worth reviewing first.

A single yes/no label is too weak for this problem. I want an ordered list of pages based on observed signals, not just a bucketed classification.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print('Section 1 framed: scoring / ranking task')


Section 1 framed: scoring / ranking task


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

I would predict a refresh-priority score for each page. A practical proxy is whether a page is both visible and declining: `trend_direction == 'down'` and `impressions_90d >= 100`.

This proxy comes from observed data, not a purely defined rule. It points at pages that already have enough traffic to matter and are losing visibility, which is the kind of page a reviewer should inspect first.


In [3]:
print('Section 2 framed: target is refresh-priority score')
proxy_definition = "trend_direction == 'down' and impressions_90d >= 100"
print('Proxy definition:', proxy_definition)


Section 2 framed: target is refresh-priority score
Proxy definition: trend_direction == 'down' and impressions_90d >= 100


## 3. Success metric

*One metric you can defend. What number means 'good'?*

A defensible metric is precision at k for the top-ranked pages, such as precision@20. That means: among the 20 pages the model ranks highest, how many match the observed declining-visible proxy?

A good number is one that is clearly higher than the baseline share of the proxy set in the full dataset. If the proxy share is around 30%, then precision@20 should be meaningfully higher than 30%.


In [4]:
print('Section 3 framed: success metric = precision@20')
print('Baseline will be computed using the dataset in section 4')


Section 3 framed: success metric = precision@20
Baseline will be computed using the dataset in section 4


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row is one content page snapshot from the starter dataset. Each row represents one page that can be refreshed, and the features are page-level signals such as impressions, trend direction, CTR, and position.

The unit of analysis is a page that a reviewer can decide to refresh or not, and the model should score these pages by refresh opportunity.


In [5]:
from pathlib import Path
import pandas as pd


def starter_csv_path():
    cwd = Path.cwd().resolve()
    search_roots = [cwd] + list(cwd.parents)

    # Common notebook and remote filesystem mounts
    extra_roots = [
        Path('/content'),
        Path('/content/drive'),
        Path('/mnt'),
        Path('/mnt/data'),
        Path('/workspace'),
        Path('/home'),
        Path('/root'),
        Path('/tmp'),
        Path.home(),
    ]
    search_roots.extend([root for root in extra_roots if root.exists()])

    excluded_roots = {'/proc', '/sys', '/dev', '/run'}
    safe_roots = [root for root in search_roots if root.is_dir() and str(root) not in excluded_roots and not str(root).startswith('/proc') and not str(root).startswith('/sys')]

    for root in safe_roots:
        candidate = root / 'data' / 'raw' / 'content_refresh_anonymized.csv'
        if candidate.exists():
            return candidate

    for root in safe_roots:
        try:
            for candidate in root.rglob('content_refresh_anonymized.csv'):
                if candidate.is_file():
                    return candidate
        except OSError:
            continue

    checked = [str(root / 'data' / 'raw' / 'content_refresh_anonymized.csv') for root in safe_roots]
    raise FileNotFoundError(
        'Starter CSV not found. Checked: ' + ', '.join(checked) +
        '; searched only common notebook/workspace roots.'
    )

path = starter_csv_path()
df = pd.read_csv(path)
proxy_label = (df['trend_direction'] == 'down') & (df['impressions_90d'] >= 100)
df = df.assign(declining_visible=proxy_label)

print('CSV path:', path)
print('Dataset shape:', df.shape)
print('One row = one page snapshot')
print('Proxy share:', round(df['declining_visible'].mean() * 100, 1), '%')
print(df.loc[:, ['declining_visible', 'impressions_90d', 'trend_direction', 'ctr', 'avg_position']].head(5))


CSV path: C:\Users\LENOVO\OneDrive\Documents\INTERNSHIP\FlyRank---Internship\data\raw\content_refresh_anonymized.csv
Dataset shape: (30000, 45)
One row = one page snapshot
Proxy share: 43.8 %
   declining_visible  impressions_90d trend_direction   ctr  avg_position
0               True             3803            down  0.76          10.6
1               True            15320            down  0.05          20.3
2               True            12581            down  0.09          36.5
3              False            11751          stable  0.49           6.2
4               True            19140            down  0.13          44.0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule would treat the same signals as separate thresholds: impressions over 100, trend down, CTR below a cutoff. In reality, the value of a page depends on how those signals combine.

A low-traffic page can still be worth reviewing if its CTR is dropping fast. A high-traffic page may be lower priority if it is stable. ML can learn those combinations and rank pages by relative refresh opportunity instead of forcing a single static rule.


In [6]:
proxy = (df['trend_direction'] == 'down') & (df['impressions_90d'] >= 100)
rule = (df['impressions_90d'] >= 500) & (df['ctr'] < df['ctr'].median())

def print_rule_comparison():
    print('Proxy pages:', int(proxy.sum()))
    print('Rule pages:', int(rule.sum()))
    print('Overlap pages:', int((proxy & rule).sum()))
    print('A fixed rule selects a different page set than the observed proxy indicates.')

print_rule_comparison()


Proxy pages: 13152
Rule pages: 4136
Overlap pages: 2742
A fixed rule selects a different page set than the observed proxy indicates.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
